## Architectural Evolution: Version 2 to Version 3
**Pushing Past 65% to Clinical-Grade AI**

### Executive Summary
Version 2 successfully established a stable training loop, eliminating architectural bugs and utilizing modern scheduling (Cosine Annealing) and augmentation (RandAugment) to achieve a highly respectable 65% accuracy. 

Version 3 (V3) transitions the codebase from a "standard classification pipeline" to a specialized, research-grade medical imaging pipeline. By drastically increasing image resolution, penalizing model overconfidence on easy edge cases, and forcing holistic feature extraction, V3 is engineered to bridge the gap toward 80%+ accuracy.

---

### 1. Information Density: Resolution & Gradient Accumulation
#### V2 Approach
Images were downscaled to `224x224` pixels with a batch size of `32`.
* **The Limitation:** 224x224 is sufficient for macroscopic objects (cats, cars, furniture), but destroys the microscopic textural details and subtle border irregularities necessary for diagnosing dermal lesions.

#### V3 Approach
Images are aggressively upscaled to `384x384`. To prevent `CUDA Out of Memory` crashes on consumer GPUs, V3 implements **Gradient Accumulation**.
* **The Engineering:** The batch size is reduced to `8` to fit in VRAM. However, the optimizer is instructed to wait and accumulate the mathematical gradients across `4` sequential batches before updating the weights. This effectively mimics a highly stable batch size of `32`, unlocking massive visual information gains without hardware crashes.

---

### 2. Advanced Regularization: Integrating CutMix
#### V2 Approach
Relied entirely on PyTorch's `RandAugment` to manipulate lighting, rotation, and shear.

#### V3 Approach
V3 stacks `CutMix` on top of RandAugment. 50% of the time, the data loader physically cuts a square patch out of one skin image (e.g., Melanoma) and pastes it directly over another (e.g., Eczema), instructing the model to predict a blended ratio of both classes.
* **The Engineering:** Skin conditions often present as overlapping clusters. CutMix prevents the model from looking for a single "anchor" feature (like one dark spot in the center) and forces the neural network to analyze the *entire* canvas, significantly improving generalization.

---

### 3. Dynamic Error Correction: Focal Loss
#### V2 Approach
Used `CrossEntropyLoss` with label smoothing. 
* **The Limitation:** CrossEntropy spends equal mathematical effort on every image. If the model is 95% confident on an "easy" obvious rash, it still adjusts weights for it, wasting compute time.

#### V3 Approach
V3 implements a custom **Focal Loss** function (with `gamma=2.0`).
* **The Engineering:** Focal Loss dynamically scales the loss based on the model's confidence. If an image is easily classified, its loss is aggressively scaled down to near zero. This forces the optimizer to redirect 100% of its learning capacity toward the difficult, borderline cases that separate human-level accuracy from AI accuracy.

---

### 4. Total Backbone Unfreezing
#### V2 Approach
During Phase 2, only the top layer stages (`model.features[5:]`) were unfrozen.

#### V3 Approach
During Phase 2, the **entire EfficientNet backbone** is unfrozen.
* **The Engineering:** Because V3 changed the input resolution to 384x384, the foundational layers (which detect basic shapes and edges) need to adjust their internal geometry to the newly scaled pixels. By utilizing an ultra-low learning rate (`1e-5`) with AdamW, we can safely retrain the entire network without experiencing "gradient shock."

---

### 5. Deployment Upgrade: Test Time Augmentation (TTA)
### V2 Approach
During validation/inference, an image was passed through the model exactly once.

#### V3 Approach
V3 includes a `predict_with_tta` inference wrapper. 
* **The Engineering:** During inference, the input image is cloned and flipped along both the horizontal and vertical axes. The model predicts all three variations, and the resulting confidence scores are averaged. This mathematically smooths out any algorithmic confusion caused by peculiar camera angles, granting an immediate accuracy boost without requiring further training.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
import os
import time
import copy

In [2]:
# ==========================================
# STEP 1. SETUP & DATA LOADING 
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

IMG_SIZE = (384, 384) 
BATCH_SIZE = 8        
ACCUMULATION_STEPS = 4 

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.RandAugment(num_ops=2, magnitude=9),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

data_dir = 'dermal_vision_dataset'
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x]) for x in ['train', 'val', 'test']}

# FIXED: num_workers=0 prevents deadlocks on Windows
dataloaders = {
    'train': DataLoader(image_datasets['train'], batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True),
    'val': DataLoader(image_datasets['val'], batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True),
    'test': DataLoader(image_datasets['test'], batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val', 'test']}
num_classes = len(image_datasets['train'].classes)

Using device: cuda


In [3]:
# ==========================================
# STEP 2. ADVANCED LOSS & CUTMIX
# ==========================================
train_targets = image_datasets['train'].targets
unique_classes = np.unique(train_targets)
weights = compute_class_weight(class_weight='balanced', classes=unique_classes, y=train_targets)
class_weights_tensor = torch.tensor(weights, dtype=torch.float).to(device)

class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.weight = weight
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

criterion = FocalLoss(weight=class_weights_tensor, gamma=2.0)

In [4]:
# ==========================================
# STEP 3. CUTMIX HELPER FUNCTION
# ==========================================
def rand_bbox(size, lam):
    W, H = size[2], size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    bbx1, bby1 = np.clip(cx - cut_w // 2, 0, W), np.clip(cy - cut_h // 2, 0, H)
    bbx2, bby2 = np.clip(cx + cut_w // 2, 0, W), np.clip(cy + cut_h // 2, 0, H)
    return bbx1, bby1, bbx2, bby2

In [9]:
# ==========================================
# STEP4. CORE TRAINING LOOP 
# ==========================================
def train_loop_v3(model, optimizer, scheduler, num_epochs, phase_name="Training"):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    
    # FIXED: Initialize the AMP Gradient Scaler
    scaler = torch.amp.GradScaler('cuda')

    for epoch in range(num_epochs):
        print(f'{phase_name} - Epoch {epoch + 1}/{num_epochs}')
        print('-' * 15)

        for phase in ['train', 'val']:
            if phase == 'train': model.train()
            else: model.eval()

            running_loss = 0.0
            running_corrects = 0
            optimizer.zero_grad()

            for i, (inputs, labels) in enumerate(dataloaders[phase]):
                inputs, labels = inputs.to(device), labels.to(device)
                
                use_cutmix = phase == 'train' and np.random.rand() < 0.5
                if use_cutmix:
                    lam = np.random.beta(1.0, 1.0)
                    rand_index = torch.randperm(inputs.size()[0]).to(device)
                    labels_a, labels_b = labels, labels[rand_index]
                    bbx1, bby1, bbx2, bby2 = rand_bbox(inputs.size(), lam)
                    inputs[:, :, bbx1:bbx2, bby1:bby2] = inputs[rand_index, :, bbx1:bbx2, bby1:bby2]
                    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (inputs.size()[-1] * inputs.size()[-2]))

                with torch.set_grad_enabled(phase == 'train'):
                    # FIXED: Wrap the forward pass in autocast to cut memory usage
                    with torch.amp.autocast('cuda'):
                        outputs = model(inputs)
                        _, preds = torch.max(outputs, 1)
                        
                        if use_cutmix:
                            loss = criterion(outputs, labels_a) * lam + criterion(outputs, labels_b) * (1. - lam)
                        else:
                            loss = criterion(outputs, labels)

                        if phase == 'train':
                            loss = loss / ACCUMULATION_STEPS

                    if phase == 'train':
                        # FIXED: Scale the loss and update using the AMP scaler
                        scaler.scale(loss).backward()
                        
                        if (i + 1) % ACCUMULATION_STEPS == 0 or (i + 1) == len(dataloaders[phase]):
                            scaler.step(optimizer)
                            scaler.update()
                            optimizer.zero_grad()

                running_loss += loss.item() * inputs.size(0) * (ACCUMULATION_STEPS if phase == 'train' else 1)
                
                if not use_cutmix or phase == 'val':
                    running_corrects += torch.sum(preds == labels.data)

            if phase == 'train' and scheduler is not None:
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]
            
            lr_str = f"| LR: {optimizer.param_groups[0]['lr']:.1e}" if phase == 'train' else ""
            print(f'{phase.capitalize():<5} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} {lr_str}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
                torch.save(model.state_dict(), 'best_dermal_model_v3.pth')
        print()

    print(f'{phase_name} complete in {(time.time() - since) // 60:.0f}m')
    print(f'Best Val Acc: {best_acc:4f}')
    model.load_state_dict(best_model_wts)
    return model

In [10]:
# ==========================================
# STEP 5. MODEL SETUP & WARMUP
# ==========================================
print("Initializing EfficientNetV2-S...")
model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)

for param in model.parameters(): param.requires_grad = False
num_ftrs = model.classifier[1].in_features
model.classifier = nn.Sequential(nn.Dropout(p=0.4), nn.Linear(num_ftrs, num_classes))
model = model.to(device)

print("\n--- PHASE 1: WARMUP HEAD ---")
optimizer_warmup = optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-2)
model = train_loop_v3(model, optimizer_warmup, scheduler=None, num_epochs=5, phase_name="Warmup")

Initializing EfficientNetV2-S...

--- PHASE 1: WARMUP HEAD ---
Warmup - Epoch 1/5
---------------
Train Loss: 2.4926 Acc: 0.1333 | LR: 1.0e-03
Val   Loss: 2.1508 Acc: 0.3207 

Warmup - Epoch 2/5
---------------
Train Loss: 2.3018 Acc: 0.1615 | LR: 1.0e-03
Val   Loss: 1.9978 Acc: 0.3377 

Warmup - Epoch 3/5
---------------
Train Loss: 2.2729 Acc: 0.1536 | LR: 1.0e-03
Val   Loss: 2.0078 Acc: 0.3316 

Warmup - Epoch 4/5
---------------
Train Loss: 2.2587 Acc: 0.1525 | LR: 1.0e-03
Val   Loss: 1.9618 Acc: 0.3485 

Warmup - Epoch 5/5
---------------
Train Loss: 2.2279 Acc: 0.1606 | LR: 1.0e-03
Val   Loss: 1.9520 Acc: 0.3446 

Warmup complete in 40m
Best Val Acc: 0.348535


In [11]:
# ==========================================
# STEP 6. DEEP FINE-TUNING
# ==========================================
print("\n--- PHASE 2: DEEP FINE-TUNING (ALL LAYERS) ---")
for param in model.parameters(): param.requires_grad = True

optimizer_ft = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)
scheduler_ft = CosineAnnealingLR(optimizer_ft, T_max=20, eta_min=1e-6)
model = train_loop_v3(model, optimizer_ft, scheduler_ft, num_epochs=20, phase_name="Fine-Tuning")


--- PHASE 2: DEEP FINE-TUNING (ALL LAYERS) ---
Fine-Tuning - Epoch 1/20
---------------
Train Loss: 2.1367 Acc: 0.1739 | LR: 9.9e-06
Val   Loss: 1.7342 Acc: 0.3830 

Fine-Tuning - Epoch 2/20
---------------
Train Loss: 2.0456 Acc: 0.1830 | LR: 9.8e-06
Val   Loss: 1.6430 Acc: 0.4050 

Fine-Tuning - Epoch 3/20
---------------
Train Loss: 1.9442 Acc: 0.2019 | LR: 9.5e-06
Val   Loss: 1.5699 Acc: 0.4177 

Fine-Tuning - Epoch 4/20
---------------
Train Loss: 1.8909 Acc: 0.1971 | LR: 9.1e-06
Val   Loss: 1.5038 Acc: 0.4358 

Fine-Tuning - Epoch 5/20
---------------
Train Loss: 1.8308 Acc: 0.2157 | LR: 8.7e-06
Val   Loss: 1.4457 Acc: 0.4428 

Fine-Tuning - Epoch 6/20
---------------
Train Loss: 1.7736 Acc: 0.2215 | LR: 8.1e-06
Val   Loss: 1.3945 Acc: 0.4609 

Fine-Tuning - Epoch 7/20
---------------
Train Loss: 1.7284 Acc: 0.2227 | LR: 7.5e-06
Val   Loss: 1.3561 Acc: 0.4654 

Fine-Tuning - Epoch 8/20
---------------
Train Loss: 1.7001 Acc: 0.2210 | LR: 6.9e-06
Val   Loss: 1.3289 Acc: 0.4678 

